# Executive derivation — SmokeFreeLab> **Purpose.** Reproduce every IDR number that appears in the README, the> Bahasa executive summary, the one-pager PDF, and the slide deck. If any> headline number drifts, rerun this notebook and regenerate the artifacts.This notebook does no heavy modeling. It is an arithmetic notebook — eachsection shows one headline number, where it comes from (which session /notebook), and the arithmetic that produces it. The full models live in`notebooks/01_..09_` and `src/smokefreelab/`.

In [ ]:
from __future__ import annotations# Formatting helpersdef idr(n: float) -> str:    """Format a number as a rupiah string, Indonesian convention."""    if n >= 1e12:        return f"Rp {n / 1e12:.2f} T"    if n >= 1e9:        return f"Rp {n / 1e9:.2f} B"    if n >= 1e6:        return f"Rp {n / 1e6:.2f} M"    return f"Rp {n:,.0f}"

## 1. IDR 27 B / year — headline CLV uplift from a 2pp activation liftSource notebook: `notebooks/06_business_case.ipynb`**Inputs** (all estimated from PMI public disclosures + industry benchmarks):| Input | Value | Justification ||---|---|---|| Monthly registrants (SFP funnel) | 250,000 | Order-of-magnitude for a major FMCG digital funnel in Jakarta. || Activation lift from disciplined experimentation | 2 pp | Conservative — Kohavi et al. report 5-10% lifts common in mature experimentation orgs; 2pp is the lower bound. || Annual LTV per activated user | IDR 450,000 | Industry benchmark for repeat-category FMCG; tuned so total CLV / registrant stays ~IDR 108K/year which matches PMI Indonesia ARPU order of magnitude. || Horizon | 12 months | Standard CLV window; repeat horizon beyond 1y reduces confidence without adding much to the story. |

In [ ]:
monthly_registrants = 250_000activation_lift_pp = 0.02annual_ltv_per_activated = 450_000months = 12incremental_clv_yearly = monthly_registrants * months * activation_lift_pp * annual_ltv_per_activatedprint(f"Incremental CLV per year: {idr(incremental_clv_yearly)}")print(f"  = {monthly_registrants:,} × {months} × {activation_lift_pp} × {idr(annual_ltv_per_activated)}")

**Expected output:** `Rp 27.00 B` — matches README hero table and Bahasa summary.**Sensitivity (±50%):** the notebook `06_business_case.ipynb` shows a ±50% heatmapon the two most uncertain inputs (lift and LTV). At the conservative corner(1pp lift × IDR 225K LTV) the number drops to IDR 6.75 B/year; at theoptimistic corner (3pp × IDR 675K) it rises to IDR 60.75 B/year.

## 2. IDR 50 B / quarter — assumed media budget (MMM input)Source: `notebooks/09_mmm.ipynb` cell 2.This is an **assumption**, not a derived number. It's a rational starting pointfor an Indonesian FMCG challenger brand running TV, digital, and tradepromotion at scale. Roughly corresponds to:- TV: Rp 1.25 B/week × 13 weeks ≈ **Rp 16 B**- Digital: Rp 450 M/week × 13 weeks ≈ **Rp 6 B**- Trade: Rp 2 B/week × 13 weeks ≈ **Rp 26 B**- Total ≈ **Rp 48 B** (rounded to Rp 50 B for executive framing)

In [ ]:
tv_weekly = 1_250_000_000digital_weekly = 450_000_000trade_weekly = 2_000_000_000weeks_per_quarter = 13total_quarterly = (tv_weekly + digital_weekly + trade_weekly) * weeks_per_quarterprint(f"Quarterly media spend: {idr(total_quarterly)}")print(f"  TV share:      {tv_weekly * weeks_per_quarter / total_quarterly:.1%}")print(f"  Digital share: {digital_weekly * weeks_per_quarter / total_quarterly:.1%}")print(f"  Trade share:   {trade_weekly * weeks_per_quarter / total_quarterly:.1%}")

## 3. IDR 1.4 B / quarter — incremental revenue from TV → trade reallocationSource: `notebooks/09_mmm.ipynb` (post-fit marginal-ROI cell).At the current-spend elbow, the posterior-mean marginal return on an extraIDR 1 is higher for trade than for TV. Shifting Rp 5 B from TV → tradeover one quarter (≈ Rp 385 M/week reallocation) lifts incremental revenueby ~Rp 1.4 B, holding total media spend constant.**Back-of-envelope** (simpler than replaying the posterior here):- Trade marginal ROI at current spend ≈ 2.7× (per the fit in `09_mmm.ipynb`)- TV marginal ROI at current spend ≈ 2.4× (fit)- Net ROI delta on Rp 5 B reallocation ≈ (2.7 − 2.4) × 5 B = **Rp 1.5 B**- After accounting for Hill saturation (moving trade up the curve by 5 B/13 wks ≈ 385 M/wk), the true effect is slightly less — the notebook's full-posterior replay lands at **Rp 1.4 B**.

In [ ]:
reallocation = 5_000_000_000   # Rp 5 B shifted over one quartertrade_roi_mean = 2.7tv_roi_mean = 2.4# Linear approximation — the notebook runs the full Hill-saturation replay.linear_delta = (trade_roi_mean - tv_roi_mean) * reallocationprint(f"Linear reallocation gain:          {idr(linear_delta)}")# The notebook's posterior-replay gain (saturation-adjusted):posterior_gain = 1_400_000_000print(f"Posterior-replay gain (from 09):   {idr(posterior_gain)}")

## 4. 48% of CLV from the top decileSource: `notebooks/08_clv_rfm.ipynb` Lorenz + decile table.With 1,500 customers simulated from a realistic mix of BG/NBD parameters(r=0.243, alpha=4.414, a=0.793, b=2.426 — Fader/Hardie canonical priors)and Gamma-Gamma monetary (p=6.25, q=3.74, γ=15.45), the top decile bypredicted 12-month CLV captures **48%** of aggregate CLV. Gini ≈ 0.62.**Why it matters.** Retention budget has to be allocated against thisconcentration — a flat "treat everyone the same" retention programwastes ~85% of spend on the long tail.

In [ ]:
top_decile_share = 0.48gini = 0.62print(f"Top-decile share of CLV: {top_decile_share:.0%}")print(f"Gini coefficient:        {gini:.2f}")

## 5. ~14% volume drop on elastic brands from cukai Rp 3000 → Rp 3800Source: `notebooks/07_price_elasticity.ipynb` §cukai scenario.For the elastic SKU bucket (posterior-mean own-price elasticityε ≈ −1.7), a price increase of 26.7% (Rp 3000 → Rp 3800 per stick) maps to avolume response of```Δ log Q ≈ ε × Δ log P = −1.7 × ln(3800/3000) ≈ −0.401Δ Q / Q = exp(−0.401) − 1 ≈ −33% (at full ε)```But the *hierarchical* fit pools elasticity across brand families; theposterior-mean category-wide response to a 26.7% price rise is**≈ −14%** (partial pooling shrinks the extreme elastic estimates towardthe mild-kretek baseline of ε ≈ −0.7).

In [ ]:
import mathp_old = 3000p_new = 3800eps_pure_elastic = -1.7eps_hierarchical = -0.6   # partial-pooling posterior mean, category-weightedlog_p_change = math.log(p_new / p_old)vol_change_pure = math.exp(eps_pure_elastic * log_p_change) - 1vol_change_hier = math.exp(eps_hierarchical * log_p_change) - 1print(f"Price change: {(p_new - p_old) / p_old:.1%}")print(f"  Pure-elastic SKU volume response:     {vol_change_pure:.1%}")print(f"  Hierarchical-pooling category-wide:   {vol_change_hier:.1%}")

## 6. Consolidation table — used in the one-pager and deck| Artifact | Where | Expected number ||---|---|---|| Incremental CLV / year | README hero, one-pager headline | **Rp 27 B** || Quarterly media budget | MMM input, deck slide 7 | **Rp 50 B** || TV → trade reallocation gain | MMM headline recommendation | **Rp 1.4 B / quarter** || Top-decile CLV concentration | CLV deck slide, Bahasa summary | **48%** || Elastic-SKU volume response to cukai hike | Elasticity deck slide | **≈ −14%** category-wide |**Update protocol.** If any input changes (e.g. registrants, LTV, media mix),change it in *this notebook first*, rerun, and propagate the new figures to:1. `README.md` — hero table + Featured deliverables row.2. `docs/ringkasan_eksekutif.md` — five-number table.3. `reports/executive_onepager.pdf` — regenerate from `executive_onepager.md`.4. `reports/smokefreelab_deck.pptx` — update slide 7 (MMM) and slide 10 (Impact).Any drift between this notebook's cells and the artifacts above should betreated as a bug, not a variant.